<a href="https://colab.research.google.com/github/Ammara-Qaisar123/AI-ML-Internship-P2/blob/main/Task_1_P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ammara Qaisar DHC-994
# Task 1
# Phase 2
This task is a News Topic Classification System using BERT.
The model classifies news articles into four categories:
World, Sports, Business, and Sci/Tech.

We used the AG News dataset and fine-tuned the pretrained
BERT model from Hugging Face Transformers library.

In [ ]:
!pip install transformers datasets evaluate scikit-learn gradio -q

# Load Dataset
The AG News dataset is a benchmark dataset for text classification.
It contains news headlines and descriptions divided into four classes.

Dataset Classes:
0 → World
1 → Sports
2 → Business
3 → Sci/Tech

Training Samples: 120,000
Testing Samples: 7,600

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
dataset

# Check Labels

In [ ]:
label_names = dataset["train"].features["label"].names
print(label_names)

# Why BERT?
BERT (Bidirectional Encoder Representations from Transformers)
is a pretrained NLP model developed by Google.

It understands context from both left and right directions,
making it highly effective for text classification tasks.

# Tokenization
Tokenization is the process of converting text into tokens
that can be understood by the BERT model.

The tokenizer converts sentences into numerical IDs,
adds special tokens, and ensures fixed input length.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define the BERT model checkpoint
checkpoint = "bert-base-uncased" # Using a common BERT base model

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Load the model with the correct number of labels from the dataset
num_labels = len(label_names)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=num_labels)

# Set id2label and label2id for the model's config using the actual label names
model.config.id2label = {i: label for i, label in enumerate(label_names)}
model.config.label2id = {label: i for i, label in enumerate(label_names)}

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Format Dataset for PyTorch

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

#Load BERT Model
# Why BERT?
BERT (Bidirectional Encoder Representations from Transformers)
is a pretrained NLP model developed by Google.

It understands context from both left and right directions,
making it highly effective for text classification tasks.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

# Model Fine Tuning
Fine-tuning means training a pretrained model on a specific dataset.

In this project, the pretrained BERT model was fine-tuned
on the AG News dataset to improve classification accuracy.

# Traning Process
The model was trained using Hugging Face Trainer API.

Training Parameters:
- Learning Rate: 2e-5
- Batch Size: 4/8/16
- Epochs: 1
- Optimizer: AdamW

A smaller subset of the dataset was used to reduce training time.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_steps=10,
)

# Evaluation Metrics
The model performance was evaluated using Accuracy and F1-score.

Accuracy:
Measures overall correct predictions.

F1-score:
Measures balance between Precision and Recall,
especially useful for classification problems.

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

# Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(500)), # Add comma here
    eval_dataset = tokenized_dataset["test"].select(range(200)),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.evaluate()

# Save Model

In [ ]:
model.save_pretrained("bert-news-model")
tokenizer.save_pretrained("bert-news-model")

#Load Model for Prediction

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="bert-news-model",
    tokenizer="bert-news-model"
)

print(classifier("Apple launches new AI chip for faster processing"))

#Deploy with Gradio
Gradio was used to create a simple web interface
for testing the news classifier.

Users can enter a news headline,
and the model predicts its category instantly.

In [ ]:
import gradio as gr

def predict(text):
    result = classifier(text)[0]
    return result

app = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="json",
    title="News Topic Classifier (BERT)"
)

app.launch()

#Challenges Faced
Main challenges faced during the project:
- Long BERT training time
- GPU memory limitations
- Dataset preprocessing
- Runtime disconnections in Google Colab

These issues were solved using smaller dataset subsets
and optimized batch sizes.

#Conclusion
The task successfully implemented a News Topic Classifier
using BERT and the AG News dataset.

The model achieved good classification performance
and demonstrated the effectiveness of transformer-based NLP models.

#Future Improvements
Future improvements may include:
- Training on larger datasets
- Increasing epochs for better accuracy
- Deploying on cloud platforms
- Using advanced transformer models like RoBERTa or DistilBERT